# Analyze and visualize PIMULTAB Eclipse table

In [2]:
# Import Python libraries
!pip install -q -U kaleido==0.2.1 -q -U skimpy

In [3]:
import os
from google.colab import drive
from pathlib import Path

# Mount Google Drive (run once per session)
drive.mount("/content/drive")

# Data directory (adjust if you move the notebook)
source = Path("/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PI/")
dest = Path("/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Output/Norne/")

# Create directory if it does not exist
os.makedirs(dest, exist_ok=True)

print("Source directory:", source)
print("Output directory:", dest)

Mounted at /content/drive
Source directory: /content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PI
Output directory: /content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Output/Norne


In [6]:
from pathlib import Path

# All entries (files + subdirs)
for p in source.iterdir():
    print(p)

# Only files
#for p in source.iterdir():
#    if p.is_file():
#        print(p)

/content/drive/MyDrive/Colab Notebooks/Petroleum Engineering/Data/Norne/INCLUDE/PI/pimultab_low-high_aug-2006.inc


## 1.0 Parse the PI table

In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

path = Path(source / "pimultab_low-high_aug-2006.inc")

fw = []
pimult = []
with path.open() as f:
    in_table = False
    for line in f:
        line = line.strip()
        if not line or line.startswith("--"):
            continue
        if line.upper().startswith("PIMULTAB"):
            in_table = True
            continue
        if not in_table:
            continue
        if line.endswith("/"):
            line = line[:-1].strip()
            if not line:
                break
        parts = line.split()
        if len(parts) >= 2:
            fw.append(float(parts[0]))
            pimult.append(float(parts[1]))

df = pd.DataFrame({"fw": fw, "PIMULT": pimult})
print(df.head())

      fw  PIMULT
0  0.000  1.0000
1  0.025  0.8341
2  0.050  0.7049
3  0.075  0.6043
4  0.100  0.5259


This reads the water-fraction fw and corresponding productivity multiplier PIMULT pairs into a DataFrame for analysis.

In [7]:
df.describe()

,fw,PIMULT
count,41.000000,41.000000
mean,0.500000,0.332690
std,0.299479,0.169501
min,0.000000,0.250000
25%,0.250000,0.250400
50%,0.500000,0.255100
75%,0.750000,0.311600
max,1.000000,1.000000


In [8]:
from skimpy import skim

# Skimpy’s main function skim(df) produces rich summary statistics for every column in a DataFrame
skim(df)

╭──────────────────────────────────────────────── skimpy summary ─────────────────────────────────────────────────╮
│          Data Summary                Data Types                                                                 │
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓ ┏━━━━━━━━━━━━━┳━━━━━━━┓                                                          │
│ ┃ Dataframe         ┃ Values ┃ ┃ Column Type ┃ Count ┃                                                          │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩ ┡━━━━━━━━━━━━━╇━━━━━━━┩                                                          │
│ │ Number of rows    │ 41     │ │ float64     │ 2     │                                                          │
│ │ Number of columns │ 2      │ └─────────────┴───────┘                                                          │
│ └───────────────────┴────────┘                                                                                  │
│                                                     number                                                      │
│ ┏━━━━━━━━━━━┳━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━┳━━━━━━━━━┓  │
│ ┃ column    ┃ NA  ┃ NA %   ┃ mean     ┃ sd       ┃ p0     ┃ p25      ┃ p50      ┃ p75      ┃ p100  ┃ hist    ┃  │
│ ┡━━━━━━━━━━━╇━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━╇━━━━━━━━━┩  │
│ │    fw     │   0 │      0 │      0.5 │   0.2995 │      0 │     0.25 │      0.5 │     0.75 │     1 │ ██▇███  │  │
│ │  PIMULT   │   0 │      0 │   0.3327 │   0.1695 │   0.25 │   0.2504 │   0.2551 │   0.3116 │     1 │   █▁    │  │
│ └───────────┴─────┴────────┴──────────┴──────────┴────────┴──────────┴──────────┴──────────┴───────┴─────────┘  │
╰────────────────────────────────────────────────────── End ──────────────────────────────────────────────────────╯

## 2.0 Plot with Plotly

In [9]:
import plotly
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"


fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=df["fw"],
        y=df["PIMULT"],
        mode="lines+markers",
        name="PIMULT(fw)"
    )
)

fig.update_layout(
    title="Eclipse PIMULTAB (low–high, Aug-2006)",
    xaxis_title="fw (water fraction)",
    yaxis_title="PIMULT",
    yaxis=dict(autorange="reversed"), # if you prefer higher PI at top, drop this
    template='ggplot2',
    height=600, width=900
)

# Define title here to be accessible for saving
plot_title = "Eclipse PIMULTAB (low–high, Aug-2006)"

# Ensure dest is a Path object for file operations
dest = Path(dest) # Explicitly convert dest to Path

plotly.offline.plot(fig, filename = str(dest / (plot_title + '.html'))) # Changed plotly.offline.plot to pyo.plot and title to plot_title
fig.write_image(str(dest / (plot_title + '.png')), scale=3.125)

fig.show()

This gives an interactive curve of PI multiplier versus water fraction for QA/QC or comparison with the analytic formula PIMULT=(1−a)/exp⁡(fw/b)+a with
a=0.25, b=0.1.

In [ ]:
import numpy as np
import pandas as pd
from scipy.optimize import curve_fit
import plotly
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "colab"

# df as built previously with columns fw, PIMULT
def pimult_model(fw, a, b):
    return (1 - a) / np.exp(fw / b) + a

p0 = [0.25, 0.1]  # initial guess from comments
popt, pcov = curve_fit(pimult_model, df["fw"].values, df["PIMULT"].values, p0=p0)

a_fit, b_fit = popt
a_err, b_err = np.sqrt(np.diag(pcov))

print(f"a = {a_fit:.6f} ± {a_err:.6f}")
print(f"b = {b_fit:.6f} ± {b_err:.6f}")

# Optional: add fitted curve to Plotly figure
fw_fit = np.linspace(df["fw"].min(), df["fw"].max(), 200)
pimult_fit = pimult_model(fw_fit, a_fit, b_fit)

fig = go.Figure()
fig.add_trace(go.Scatter(x=df["fw"], y=df["PIMULT"],
                         mode="markers", name="Data"))
fig.add_trace(go.Scatter(x=fw_fit, y=pimult_fit,
                         mode="lines", name=f"Fit: a={a_fit:.4f}, b={b_fit:.4f}"))

fig.update_layout(
    title="Eclipse PIMULTAB (low–high, Aug-2006) and fitted curve",
    xaxis_title="fw (water fraction)",
    yaxis=dict(autorange="reversed"), # if you prefer higher PI at top, drop this
    yaxis_title="PIMULT",
    template='ggplot2',
    height=600, width=900,
    legend=dict(
    x=1.0,
    y=0.0,
    xanchor="right",
    yanchor="bottom",
    bgcolor="rgba(255,255,255,0.8)"
    )
)

# Define title here to be accessible for saving
plot_title = "Eclipse PIMULTAB (low–high, Aug-2006) and fitted curve"

# Ensure dest is a Path object for file operations
dest = Path(dest) # Explicitly convert dest to Path

plotly.offline.plot(fig, filename = str(dest / (plot_title + '.html')))
fig.write_image(str(dest / (plot_title + '.png')), scale=3.125)

fig.show()

a = 0.249998 ± 0.000006
b = 0.099996 ± 0.000005
